In [1]:
import pandas as pd
import re
import json
import swifter
from langchain.text_splitter import RecursiveCharacterTextSplitter




def before_chunk(text: str) -> str:
    text = re.sub(r"\s*Văn\s*bản\s*này\s*chưa\s*cập\s*nhật\s*nội\s*dung\s*Tiếng\s*Anh\s*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"[\\/*?\"<>|]", "", text)
    text = re.sub(r"\t+", " ", text)
    text = re.sub(r"[\[\];']", " ", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"^[ ]+|[ ]+$", "", text, flags=re.MULTILINE)
    return text.strip()


def after_chunk(text: str) -> str:
    text = re.sub(r"[\n]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def hierarchical_law_chunker(text: str, max_tokens: int = 1000):
    """
    Chunk văn bản luật theo phân cấp thông minh:
    - Luật -> Chương -> Điều
    - Luôn tôn trọng ranh giới tự nhiên
    - Chỉ cắt khi gặp cấp tương đương hoặc cao hơn
    """
    
    # Pattern cải tiến cho văn bản luật Việt Nam
    patterns = {
        "law": r"^(LUẬT|NGHỊ QUYẾT|PHÁP LỆNH)\s+.+?(?=\n|$)",
        "chapter": r"^(Chương\s+[IVXLCDM]+|[IVXLCDM]+\.).*?(?=\n|$)",
        "article": r"^(Điều\s+\d+\.?).*?(?=\n|$)",
        "part": r"^(Phần\s+[A-Z]).*?(?=\n|$)"
    }
    
    def extract_hierarchical_structure(text):
        """Trích xuất cấu trúc phân cấp từ văn bản"""
        lines = text.split('\n')
        structure = []
        current_level = "content"
        current_content = []
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            # Xác định cấp độ của dòng hiện tại
            if re.match(patterns["law"], line, re.IGNORECASE | re.MULTILINE):
                level = "law"
            elif re.match(patterns["part"], line, re.IGNORECASE | re.MULTILINE):
                level = "part" 
            elif re.match(patterns["chapter"], line, re.IGNORECASE | re.MULTILINE):
                level = "chapter"
            elif re.match(patterns["article"], line, re.IGNORECASE | re.MULTILINE):
                level = "article"
            else:
                level = "content"
            
            # Nếu gặp cấp mới, lưu nội dung cũ và bắt đầu cấp mới
            if level != "content" and current_content:
                structure.append({
                    "level": current_level,
                    "content": "\n".join(current_content)
                })
                current_content = []
                current_level = level
            
            current_content.append(line)
        
        # Thêm phần cuối cùng
        if current_content:
            structure.append({
                "level": current_level,
                "content": "\n".join(current_content)
            })
            
        return structure

    def build_chunks_from_structure(structure, max_tokens):
        """Xây dựng chunks từ cấu trúc phân cấp"""
        chunks = []
        current_chunk = ""
        current_chunk_tokens = 0
        
        for i, element in enumerate(structure):
            element_content = element["content"]
            element_tokens = len(element_content.split())
            
            # QUY TẮC QUAN TRỌNG: Chỉ cắt chunk khi:
            # 1. Gặp Luật/Phần mới (cấp cao nhất)
            # 2. Gặp Chương mới VÀ chunk hiện tại đã đủ lớn
            # 3. Gặp Điều mới VÀ chunk hiện tại + Điều mới vượt max_tokens
            
            should_break = False
            
            # Luật/Phần mới -> luôn cắt
            if element["level"] in ["law", "part"] and current_chunk:
                should_break = True
            
            # Chương mới -> cắt nếu chunk hiện tại đã có nội dung đáng kể
            elif element["level"] == "chapter" and current_chunk_tokens > max_tokens * 0.3:
                should_break = True
            
            # Điều mới -> cắt nếu thêm vào sẽ vượt quá max_tokens
            elif element["level"] == "article" and current_chunk_tokens + element_tokens > max_tokens:
                should_break = True
            
            # Cắt chunk nếu cần
            if should_break and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""
                current_chunk_tokens = 0
            
            # Thêm phần tử hiện tại vào chunk
            if current_chunk:
                current_chunk += "\n" + element_content
            else:
                current_chunk = element_content
            current_chunk_tokens += element_tokens
            
            # Đảm bảo không vượt quá max_tokens quá nhiều
            if current_chunk_tokens > max_tokens * 1.2:
                # Trường hợp hiếm: một Điều quá dài, phải chia nhỏ
                if len(current_chunk.split()) > max_tokens:
                    split_chunks = split_oversized_element(current_chunk, max_tokens)
                    if len(split_chunks) > 1:
                        chunks.extend(split_chunks[:-1])
                        current_chunk = split_chunks[-1]
                        current_chunk_tokens = len(current_chunk.split())
                    else:
                        chunks.append(current_chunk.strip())
                        current_chunk = ""
                        current_chunk_tokens = 0
        
        # Thêm chunk cuối cùng
        if current_chunk:
            chunks.append(current_chunk.strip())
            
        return chunks

    def split_oversized_element(element, max_tokens):
        """Chia nhỏ các phần tử quá lớn (Điều quá dài)"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=max_tokens,
            chunk_overlap=50,
            separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""]
        )
        return splitter.split_text(element)

    # XỬ LÝ CHÍNH
    structure = extract_hierarchical_structure(text)
    chunks = build_chunks_from_structure(structure, max_tokens)
    
    # Đảm bảo không chunk nào vượt quá max_tokens quá nhiều
    final_chunks = []
    for chunk in chunks:
        if len(chunk.split()) > max_tokens * 1.1:
            sub_chunks = split_oversized_element(chunk, max_tokens)
            final_chunks.extend(sub_chunks)
        else:
            final_chunks.append(chunk)
    
    return final_chunks

def law_chunker_with_metadata(text: str, max_tokens: int = 1000):
    """
    Phiên bản có metadata cho search engine
    """
    chunks = hierarchical_law_chunker(text, max_tokens)
    
    chunks_with_metadata = []
    for i, chunk in enumerate(chunks):
        # Extract thông tin từ chunk
        metadata = extract_chunk_metadata(chunk, i)
        chunks_with_metadata.append({
            "content": chunk,
            "metadata": metadata
        })
    
    return chunks_with_metadata

def extract_chunk_metadata(chunk: str, chunk_id: int):
    """Trích xuất metadata từ chunk"""
    metadata = {
        "chunk_id": chunk_id,
        "token_count": len(chunk.split()),
        "char_count": len(chunk),
        "hierarchy_levels": []
    }
    
    # Xác định các cấp độ có trong chunk
    patterns = {
        "law": r"^(LUẬT|NGHỊ QUYẾT|PHÁP LỆNH)\s+(.+?)(?=\n|$)",
        "chapter": r"^Chương\s+([IVXLCDM]+).*?(?=\n|$)",
        "article": r"^Điều\s+(\d+).*?(?=\n|$)"
    }
    
    lines = chunk.split('\n')
    for line in lines:
        line = line.strip()
        for level, pattern in patterns.items():
            match = re.search(pattern, line, re.IGNORECASE | re.MULTILINE)
            if match:
                if level == "law":
                    metadata["law_title"] = line
                elif level == "chapter":
                    metadata["chapter"] = match.group(1)
                elif level == "article":
                    if "articles" not in metadata:
                        metadata["articles"] = []
                    metadata["articles"].append(match.group(1))
                metadata["hierarchy_levels"].append(level)
                break
    
    # Remove duplicates
    metadata["hierarchy_levels"] = list(set(metadata["hierarchy_levels"]))
    
    return metadata



ModuleNotFoundError: No module named 'langchain.text_splitter'

In [37]:
data = "SẮC LỆNH\n\nCỦA CHỦ TỊCH NƯỚC VIỆT NAM DÂN CHỦ CỘNG HOÀ SỐ 68/SL NGÀY 18 THÁNG 6 NĂM ẤN ĐỊNH KẾ HOẠCH THỰC HÀNH CÁC CÔNG TÁC THUỶ NÔNG VÀ THỂ LỆ BẢO VỆ CÁC CÔNG TRÌNH THUỶ NÔNG949\n\nCHỦ TỊCH NƯỚC VIỆT NAM DÂN CHỦ CỘNG HOÀ\n\nChiểu Sắc lệnh số 194-SL ngày 28 tháng 5 năm 1948 thành lập các Uỷ ban bảo vệ đê điều;\n\nChiểu Sắc lệnh số 104-SL ngày 1 tháng 1 năm 1948 quy lệ các doanh nghiệp quốc gia;\n\nTheo đề nghị của các Bộ trưởng Bộ Giao thông Công chính, Bộ Nội vụ, Bộ Tư pháp, Bộ Canh nông;\n\nTheo quyết nghị của Hội đồng Chính phủ sau khi Ban Thường trực Quốc hội thoả thuận;\n\nRA SẮC LỆNH:\n\nĐiều 1\n\nSắc lệnh này ấn định:\n\n- Kế hoạch thực hành các công tác thuỷ nông,\n\n- Thể lệ bảo vệ các công trình thuỷ nông.\n\nCông tác thuỷ nông là những công tác cần thiết để tăng hoa lợi ruộng đất, tránh nạn mất mùa và bảo toàn sinh mạng và tài sản của nhân dân, bằng cách điều hoà và sử dụng các nguồn nước thiên nhiên, như dẫn nước tưới ruộng, rút nước thừa ở ruộng, ngăn nước lụt, chắn nước mặn, thau phèn chua trong đồng, vân, vân ...\n\nCác công trình huỷ nông gồm có đê, đập, cầu cống, kè, máy bơm, kênh, mương và tất cả những công trình thuỷ lợi có mục đích tích trữ, phân phối, điều hoà, lưu thông hay ngăn cản dòng nước, để làm lợi cho nghề nông.\n\nĐiều 2\n\nCác công tác tu bổ và hộ đê thường niên do các Uỷ ban bảo vệ đê điều các cấp liên khu, tỉnh, huyện, xã thành lập theo Sắc lệnh số 194-SL ngày 28 tháng 5 năm 1948 phụ trách.\n\nCHƯƠNG 1\n\nCÁC HỘI ĐỒNG THUỶ NÔNG\n\na) Cấp tỉnh\n\nĐiều 3\n\nNay lập ở mỗi tỉnh một hội đồng thuỷ nông gồm có:\n\nChủ tịch UBKCHC tỉnh hay người đại diện Chủ tịch\n\nTrưởng Ty Túc mễ hay Khuyến nông do UBKCHC tỉnh cử Thư ký\n\nTrưởng Ty Công chính tỉnh Thuyết trình viên\n\nMột đại biểu đoàn thể nông dân, do đoàn thể cử Uỷ viên\n\nMột đại biểu nông gia, do Hội đồng nhân dân tỉnh cử Uỷ viên\n\nĐiều 4\n\nNhiệm vụ của Hội đồng thuỷ nông tỉnh là:\n\n1- Xét các công tác thuỷ nông, kể cả đê điều, trong tỉnh, và đề nghị cùng các cơ quan chuyên môn nghiên cứu và thực hành.\n\n2- Vận động nhân dân tham gia bằng cách giúp đỡ của và công vào việc xây dựng, tu bổ và khai thác các công trình thuỷ nông sau khi đã được UBKCHC tỉnh thoả thuận.\n\nb) Cấp liên khu\n\nĐiều 5\n\nNay lập ở mỗi liên khu một Hội đồng thuỷ nông gồm có:\n\nChủ tịch UBKCHC liên khu hay người đại diện Chủ tịch\n\nGiám đốc Nông chính liên khu Thư ký\n\nGiám đốc công chính liên khu Thuyết trình viên\n\nMột đại biểu đoàn thể nông dân do đoàn thể cử Uỷ viên\n\nMột đại biểu nông gia do UBKCHC liên khu cử Uỷ viên\n\nĐiều 6\n\nNhiệm vụ của Hội đồng thuỷ nông liên khu là:\n\n1- Xét các đề nghị và kiểm soát các công việc của các Hội đồng thuỷ nông tỉnh;\n\n2- Nghiên cứu và đề nghị thực hành những công tác lợi ích cho nhiều tỉnh trong liên khu, vận động dân chúng tham gia bằng cách góp của và công vào việc xây dựng, tu bổ và khai thác các công tác thuỷ nông sau khi đã hỏi ý kiến các Hội đồng thuỷ nông và UBKCHC các tỉnh sở quản.\n\nCHƯƠNG 2\n\nCÁC PHẠM PHÁP\n\nĐiều 7\n\nCấm không ai được đào đất, trồng cây, cắm cọc, làm nhà, cho súc vật dẫm phá gần đê, đập, kênh và cầu cống phụ thuộc, trong một địa phận bảo vệ, do Bộ Giao thông Công chính ấn định ; hoặc làm hư hỏng, bằng một cách nào khác, các công trình thuỷ nông. Chỉ những nhân viên chuyên môn chuyên trách mới được phép sử dụng các công trình thuỷ nông, theo đúng mục đích của các công trình đó.\n\nĐiều 8\n\nNhững người phạm vào điều 7 trên này sẽ bị phạt như sau:\n\n- Phạt tiền từ 100đ đến 100.000đ\n\n- Phạt tù:\n\n+ Từ 10 ngày đến 2 năm, nếu việc phạm pháp gây thiệt hại cho nhân dân trong một xã;\n\n+ Từ 3 tháng đến 5 năm, nếu thiệt hại cho nhân dân một huyện;\n\n+ Từ 1 năm đến chung thân, nếu thiệt hại cho nhân dân trong một tỉnh;\n\n+ Từ 3 năm đến tử hình, nếu thiệt hại cho nhân dân nhiều tỉnh;\n\n- Hoặc một trong hai hình phạt trên.\n\nNgoài ra, can phạm còn phải bồi thường để sửa chữa những sự hư hỏng đã làm ra. Số tiền bồi thường sẽ do sở Công chính ấn định.\n\nTrong thời kỳ kháng chiến, những tội kể trên có thể bị truy tố trước toà án quân sự.\n\nĐiều 9\n\nĐược lập biên bản các phạm pháp:\n\n- Các uỷ viên trong UBKCHC các cấp ;\n\n- Các Ban tư pháp xã, các phụ trách và uỷ viên tư pháp Công an ;\n\n- Các nhân viên Nông giang chuyên trách, từ cấp cán sự trở lên;\n\n- Các nhân viên Canh nông chuyên trách, từ cấp cán sự trở lên.\n\nĐiều 10\n\nCác Bộ trưởng Bộ Nội vụ, Bộ Tư pháp, Bộ Giao thông Công chính và Bộ Canh nông, mỗi khi cần, sẽ ra nghị định quy định, trong phạm vi mỗi Bộ, các chi tiết để thi hành sắc lệnh này.\n\nĐiều 11\n\nCác thể lệ trước đây trái với sắc lệnh này đều bãi bỏ.\n\nĐiều 12\n\nCác ông Bộ trưởng Bộ Nội vụ, Bộ Tư pháp, Bộ giao thông Công chính và Bộ Canh nông, chiểu sắc lệnh thi hành.\n\n \n\n\t\n\nHồ Chí Minh\n\n(Đã ký)\n\nVăn bản này chưa cập nhật nội dung Tiếng Anh"  
test = before_chunk(data)
print(repr(test))
print(test)



'SẮC LỆNH\n\nCỦA CHỦ TỊCH NƯỚC VIỆT NAM DÂN CHỦ CỘNG HOÀ SỐ 68SL NGÀY 18 THÁNG 6 NĂM ẤN ĐỊNH KẾ HOẠCH THỰC HÀNH CÁC CÔNG TÁC THUỶ NÔNG VÀ THỂ LỆ BẢO VỆ CÁC CÔNG TRÌNH THUỶ NÔNG949\n\nCHỦ TỊCH NƯỚC VIỆT NAM DÂN CHỦ CỘNG HOÀ\n\nChiểu Sắc lệnh số 194-SL ngày 28 tháng 5 năm 1948 thành lập các Uỷ ban bảo vệ đê điều\n\nChiểu Sắc lệnh số 104-SL ngày 1 tháng 1 năm 1948 quy lệ các doanh nghiệp quốc gia\n\nTheo đề nghị của các Bộ trưởng Bộ Giao thông Công chính, Bộ Nội vụ, Bộ Tư pháp, Bộ Canh nông\n\nTheo quyết nghị của Hội đồng Chính phủ sau khi Ban Thường trực Quốc hội thoả thuận\n\nRA SẮC LỆNH:\n\nĐiều 1\n\nSắc lệnh này ấn định:\n\n- Kế hoạch thực hành các công tác thuỷ nông,\n\n- Thể lệ bảo vệ các công trình thuỷ nông.\n\nCông tác thuỷ nông là những công tác cần thiết để tăng hoa lợi ruộng đất, tránh nạn mất mùa và bảo toàn sinh mạng và tài sản của nhân dân, bằng cách điều hoà và sử dụng các nguồn nước thiên nhiên, như dẫn nước tưới ruộng, rút nước thừa ở ruộng, ngăn nước lụt, chắn nước mặn,

In [39]:
chunkded = law_chunker(test, max_tokens=100)
for i, c in enumerate(chunkded):
    print(f"--- Chunk {i+1} ---")
    print(repr(c[:200]))  # dùng repr để thấy rõ \n
    print()

AttributeError: type object 'builtins.TextSplitter' has no attribute 'from_tiktoken_encoder'

In [4]:
df = pd.read_json("tvpl_congvan.json")
df.head()

,title,href,lawid,content,mt6_text,date
0,Nghị định năm 1945 về Hội đồng cố vấn học chín...,https://thuvienphapluat.vn/van-ban/Giao-duc/Ng...,22714,Giáo dục,BỘ QUỐC GIA GIÁO DỤC\n******\n\n\t\n\nVIỆT NAM...,1945-10-15
1,Nghị định về tiền hoa hồng đăng ký vào ngân sá...,https://thuvienphapluat.vn/van-ban/Tai-chinh-n...,22504,Tài chính nhà nước,BỘ TÀI CHÍNH\n******\n\n\t\n\nVIỆT NAM DÂN CHỦ...,1945-10-15
2,Sắc lệnh số 51 năm 1945 về việc ấn định thể lệ...,https://thuvienphapluat.vn/van-ban/Bo-may-hanh...,35900,Bộ máy hành chính,SẮC LỆNH\n\nCỦA CHỦ TỊCH NƯỚC SỐ 51 NGÀY 17 TH...,1945-10-17
3,Sắc lệnh số 52 về việc xá tội các phạm nhân do...,https://thuvienphapluat.vn/van-ban/Thu-tuc-To-...,35902,Thủ tục Tố tụng,SẮC LỆNH\n\nCỦA CHỦ TỊCH CHÍNH PHỦ LÂM THỜI SỐ...,1945-10-20
4,Sắc lệnh số 53 về việc quy định quốc tịch Việt...,https://thuvienphapluat.vn/van-ban/Quyen-dan-s...,35901,Quyền dân sự,SẮC LỆNH\n\nCỦA CHỦ TỊCH CHÍNH PHỦ LÂM THỜI SỐ...,1945-10-20


In [10]:
df_clean = pd.read_json("laws_luatVN_chunk.json")
df_clean["content_chunk"] = df_clean["content_chunk"].swifter.apply(sanitize_content)
df_clean.to_json("clean_luatVN_chunk.json", orient="records", force_ascii=False, indent=2)

Pandas Apply: 100%|██████████| 33354/33354 [00:02<00:00, 12576.52it/s]


In [ ]:
df1 = pd.read_json("laws_luatVN_chunk.json")
df1.head(10)

,content,auto_id,law_id
0,,1,None
1,BỘ CÔNG THƯƠNG ------- CỘNG HÒA XÃ HỘI CHỦ NGH...,2,7149/CĐ-BCT
2,UBND THÀNH PHỐ HÀ NỘI TRUNG TÂM PHỤC VỤ HÀNH C...,3,1329/QĐ-TTPVHCC
3,VĂN PHÒNG CHÍNH PHỦ -------- CỘNG HÒA XÃ HỘI C...,4,505/TB-VPCP
4,VĂN PHÒNG CHÍNH PHỦ -------- CỘNG HÒA XÃ HỘI C...,5,503/TB-VPCP
5,THỦ TƯỚNG CHÍNH PHỦ ------- CỘNG HÒA XÃ HỘI CH...,6,170/CĐ-TTg
6,CHÍNH PHỦ ------- CỘNG HÒA XÃ HỘI CHỦ NGHĨA VI...,7,66.4/2025/NQ-CP
7,BAN CHỈ ĐẠO VỀ TỔNG KẾT THỰC HIỆN NGHỊ QUYẾT S...,8,130/KH-BCĐTKNQ18
8,THỦ TƯỚNG CHÍNH PHỦ ------- CỘNG HÒA XÃ HỘI CH...,9,2109/QĐ-TTg
9,THỦ TƯỚNG CHÍNH PHỦ ------- CỘNG HÒA XÃ HỘI CH...,10,169/CĐ-TTg
